In [1]:
import pandas as pd

In [2]:
patients = pd.read_csv("patients.csv")
encounters = pd.read_csv("encounters.csv")
procedures = pd.read_csv("procedures.csv")
payers = pd.read_csv("payers.csv")
organizations = pd.read_csv("organizations.csv")

In [3]:
print("Patients Data:")
print(patients.head())
print("\nEncounters Data:")
print(encounters.head())

Patients Data:
                                     Id   BIRTHDATE   DEATHDATE PREFIX  \
0  5605b66b-e92d-c16c-1b83-b8bf7040d51f  1977-03-19         NaN   Mrs.   
1  6e5ae27c-8038-7988-e2c0-25a103f01bfa  1940-02-19         NaN    Mr.   
2  8123d076-0886-9007-e956-d5864aa121a7  1958-06-04         NaN    Mr.   
3  770518e4-6133-648e-60c9-071eb2f0e2ce  1928-12-25  2017-09-29    Mr.   
4  f96addf5-81b9-0aab-7855-d208d3d352c5  1928-12-25  2014-02-23    Mr.   

       FIRST           LAST SUFFIX     MAIDEN MARITAL   RACE    ETHNICITY  \
0  Nikita578      Erdman779    NaN  Leannon79       M  white  nonhispanic   
1    Zane918  Hodkiewicz467    NaN        NaN       M  white  nonhispanic   
2   Quinn173   Marquardt819    NaN        NaN       M  white  nonhispanic   
3    Abel832     Smitham825    NaN        NaN       M  white     hispanic   
4   Edwin773     Labadie908    NaN        NaN       M  white     hispanic   

  GENDER                    BIRTHPLACE                       ADDRESS    CITY 

In [6]:
# Rename columns for easier merging
patients = patients.rename(columns={'Id':'patient_id'})
encounters = encounters.rename(columns={
    'Id':'encounter_id',
    'PATIENT':'patient_id',
    'ORGANIZATION':'organization_id',
    'PAYER':'payer_id',
    'START':'admission_date',
    'STOP':'discharge_date',
    'BASE_ENCOUNTER_COST':'base_encounter_cost',
    'TOTAL_CLAIM_COST':'total_claim_cost',
    'PAYER_COVERAGE':'payer_coverage',
    'DESCRIPTION':'encounter_description'
})
procedures = procedures.rename(columns={
    'ENCOUNTER':'encounter_id',
    'PATIENT':'patient_id',
    'START':'procedure_start',
    'STOP':'procedure_stop',
    'DESCRIPTION':'procedure_description',
    'BASE_COST':'procedure_cost'
})
payers = payers.rename(columns={'Id':'payer_id', 'NAME':'payer_name'})
organizations = organizations.rename(columns={'Id':'organization_id', 'NAME':'organization_name'})

In [7]:
# Merge encounters with patients
enc_pat = encounters.merge(patients, how='left', on='patient_id')

# Merge with procedures
enc_pat_proc = enc_pat.merge(procedures, how='left', on=['encounter_id','patient_id'])

# Merge with payers
full_data = enc_pat_proc.merge(payers, how='left', on='payer_id')

# Merge with organizations
full_data = full_data.merge(organizations, how='left', on='organization_id')

# Convert dates to datetime
full_data['admission_date'] = pd.to_datetime(full_data['admission_date'])
full_data['discharge_date'] = pd.to_datetime(full_data['discharge_date'])
full_data['procedure_start'] = pd.to_datetime(full_data['procedure_start'])
full_data['procedure_stop'] = pd.to_datetime(full_data['procedure_stop'])

# Calculate length of stay
full_data['length_of_stay'] = (full_data['discharge_date'] - full_data['admission_date']).dt.days

full_data.head()

,encounter_id,admission_date,discharge_date,patient_id,organization_id,payer_id,ENCOUNTERCLASS,CODE_x,encounter_description,base_encounter_cost,...,ZIP_y,PHONE,organization_name,ADDRESS,CITY,STATE_y,ZIP,LAT_y,LON_y,length_of_stay
0,32c84703-2481-49cd-d571-3899d5820253,2011-01-02 09:26:36+00:00,2011-01-02 12:58:36+00:00,3de74169-7f67-9304-91d4-757e0f3a14d2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,ambulatory,185347001,Encounter for problem (procedure),85.55,...,NaN,NaN,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,2114,42.362813,-71.069187,0
1,c98059da-320a-c0a6-fced-c8815f3e3f39,2011-01-03 05:44:39+00:00,2011-01-03 06:01:42+00:00,d9ec2e44-32e9-9148-179a-1653348cc4e2,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,outpatient,308335008,Patient encounter procedure,142.58,...,NaN,NaN,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,2114,42.362813,-71.069187,0
2,4ad28a3a-2479-782b-f29c-d5b3f41a001e,2011-01-03 14:32:11+00:00,2011-01-03 14:47:11+00:00,73babadf-5b2b-fee7-189e-6f41ff213e01,d78e84ec-30aa-3bba-a33a-f29a3a454662,7caa7254-5050-3b5e-9eae-bd5ea30e809c,outpatient,185349003,Encounter for check up (procedure),85.55,...,21244.0,1-800-633-4227,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,2114,42.362813,-71.069187,0
3,c3f4da61-e4b4-21d5-587a-fbc89943bc19,2011-01-03 16:24:45+00:00,2011-01-03 16:39:45+00:00,3b46a0b7-0f34-9b9a-c319-ace4a1f58c0b,d78e84ec-30aa-3bba-a33a-f29a3a454662,b1c428d6-4f07-31e0-90f0-68ffa6ff8c76,wellness,162673000,General examination of patient (procedure),136.80,...,NaN,NaN,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,2114,42.362813,-71.069187,0
4,a9183b4f-2572-72ea-54c2-b3cd038b4be7,2011-01-03 17:36:53+00:00,2011-01-03 17:51:53+00:00,fa006887-d93c-d302-8b89-f3c25f88c0e1,d78e84ec-30aa-3bba-a33a-f29a3a454662,42c4fca7-f8a9-3cd1-982a-dd9751bf3e2a,ambulatory,390906007,Follow-up encounter,85.55,...,46204.0,1-800-331-1476,MASSACHUSETTS GENERAL HOSPITAL,55 FRUIT STREET,BOSTON,MA,2114,42.362813,-71.069187,0


In [8]:
# Average patient length of stay
avg_los = full_data['length_of_stay'].mean()
print("Average Length of Stay:", avg_los)

# Total cost by payer
cost_by_payer = full_data.groupby('payer_name')['total_claim_cost'].sum().reset_index()
print(cost_by_payer)

# Top 10 procedures
top_procedures = full_data['procedure_description'].value_counts().head(10)
print(top_procedures)

# Patient visits by organization
visits_by_org = full_data.groupby('organization_name')['encounter_id'].count().reset_index()
print(visits_by_org)

Average Length of Stay: 0.24958143199501
               payer_name  total_claim_cost
0                   Aetna      1.428996e+07
1                  Anthem      2.289787e+07
2  Blue Cross Blue Shield      1.660025e+07
3            Cigna Health      1.547713e+07
4           Dual Eligible      7.903969e+06
5                  Humana      1.865669e+07
6                Medicaid      5.962049e+07
7                Medicare      9.137077e+07
8            NO_INSURANCE      1.400480e+08
9        UnitedHealthcare      1.334011e+07
procedure_description
Assessment of health and social care needs (procedure)                                4596
Hospice care (regime/therapy)                                                         4098
Depression screening using Patient Health Questionnaire Two-Item score (procedure)    3614
Depression screening (procedure)                                                      3614
Assessment of substance use (procedure)                                               290

In [9]:
full_data.to_csv("healthcare_analytics_cleaned.csv", index=False)
print("Dataset ready for Power BI dashboard!")

Dataset ready for Power BI dashboard!
